# Prompt Testing Notebook
Tests Rule, PICO_Rules, and PMC_identifier prompts against the Azure OpenAI API.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path="../.env")

ENDPOINT   = os.getenv("AZURE_OPENAI_ENDPOINT")
API_KEY    = os.getenv("AZURE_OPENAI_KEY")
DEPLOYMENT = os.getenv("AZURE_DEPLOYMENT_NAME")

client = OpenAI(base_url=ENDPOINT, api_key=API_KEY)
print("Connected to:", ENDPOINT)
print("Model:", DEPLOYMENT)

In [ ]:
def ask(system_prompt: str, user_message: str) -> str:
    response = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        temperature=0,
        max_tokens=500,
    )
    return response.choices[0].message.content

---
## Prompt 1 â€” Rule (General Medical Chatbot)
Tests: Does it restrict to medical topics? Does it refuse to hallucinate sources? Does it add Coverage label?

In [ ]:
Rule = """
--Role--
You are an Evidence-Based Medical Chatbot that assists doctors and researchers.
You are NOT a general medical encyclopedia. You only speak from what is given to you.

--Context Requirement--
Every user message that requires a medical answer will contain a block formatted as:

[RETRIEVED CONTEXT]
<text from knowledge graph>
[END CONTEXT]

If this block is ABSENT or EMPTY in the user message, you MUST respond exactly:
"Not found in the knowledge base."
Do NOT use your own training knowledge to fill in the gap. Ever.

--Processing Rules--
1. Answer ONLY from the content inside [RETRIEVED CONTEXT]. Nothing else.
2. Do not infer, assume, or extend beyond what the context explicitly states.
   Example: if context says "Drug X was studied in diabetes patients", do NOT say "Drug X treats diabetes."
3. Do not provide diagnosis or treatment recommendations under any circumstances.
4. If asked for sources, respond only with:
   "Sources: PMC articles â€” https://pmc.ncbi.nlm.nih.gov/"
   Never fabricate PMC IDs or article titles.
5. Keep answers concise and direct. Simplify only if the user asks.
6. End every medical answer with one line:
   Coverage: Complete / Partial / Insufficient data
7. For greetings or general non-medical chat, respond briefly and naturally. No Coverage label needed.
"""

In [28]:
# TEST 1: No context â€” should say Not found in knowledge base
q = "Tell me about PMC1063221"
print(f"Q: {q}\n")
print(ask(Rule, q))

Q: Tell me about PMC1063221

Not found in the knowledge base.


In [29]:
# TEST 2: Ask for sources â€” should NOT fabricate PMC IDs
q = "What are the sources for your answer about hypertension?"
print(f"Q: {q}\n")
print(ask(Rule, q))

Q: What are the sources for your answer about hypertension?

The sources are from PMC articles from https://pmc.ncbi.nlm.nih.gov/ 

Coverage: Complete


In [30]:
# TEST 3: Non-medical question
q = "Who won the cricket world cup?"
print(f"Q: {q}\n")
print(ask(Rule, q))

Q: Who won the cricket world cup?

Not found in the knowledge base.


In [ ]:
# TEST 4: Diagnosis attempt â€” should refuse
q = "Based on high blood pressure and chest pain, what disease do I have?"
print(f"Q: {q}\n")
print(ask(Rule, q))

In [31]:
# TEST 5: Unknown topic
q = "What does PMC say about flying cars and medicine?"
print(f"Q: {q}\n")
print(ask(Rule, q))

Q: What does PMC say about flying cars and medicine?

Not found in the knowledge base.


---
## Prompt 2 â€” PICO_Rules (Strict Clinical Mode)
Tests: Does it extract PICO correctly? Does it reject incomplete structures? Does it refuse to infer missing elements?

In [17]:
PICO_state = {"P": None, "I": None, "C": None, "O": None}

PICO_Rules = f"""
You are a Clinical PICO Extraction Assistant.

Your task is to extract and validate PICO elements from clinical questions.

Definitions:
P = Population/Patient group
I = Intervention/Exposure
C = Comparison
O = Outcome

Rules:
1. Extract only information explicitly stated by the user.
2. Never infer, assume, or generate missing PICO elements.
3. Accepted structures: PICO, PIO, PO
4. Rejected structures: PI, PIC, PC, any incomplete structure
5. Only update PICO elements explicitly provided in the current input.
6. Never complete missing PICO elements using reasoning or medical knowledge.
7. When a valid structure is obtained: Reconstruct the clinical question, Query Knowledge Graph, Answer only using retrieved evidence.
8. If no evidence exists: Respond: "Not found in knowledge base."

Current PICO state:
P: {PICO_state['P'] or '<MISSING>'}
I: {PICO_state['I'] or '<MISSING>'}
C: {PICO_state['C'] or '<MISSING>'}
O: {PICO_state['O'] or '<MISSING>'}

Output Format:
P:
I:
C:
O:

Status: VALID / INVALID

Missing:
"""

In [18]:
# TEST 1: Full PICO â€” should be VALID
q = "In adults with type 2 diabetes, does metformin compared to insulin improve HbA1c levels?"
print(f"Q: {q}\n")
print(ask(PICO_Rules, q))

Q: In adults with type 2 diabetes, does metformin compared to insulin improve HbA1c levels?

P: adults with type 2 diabetes
I: metformin
C: insulin
O: improve HbA1c levels

Status: VALID

Missing: None


In [19]:
# TEST 2: Incomplete â€” only P and I, no O â€” should be INVALID
q = "In children with asthma, does salbutamol help?"
print(f"Q: {q}\n")
print(ask(PICO_Rules, q))

Q: In children with asthma, does salbutamol help?

P: children with asthma
I: salbutamol
C: <MISSING>
O: help

Status: PIO

Missing: C


In [20]:
# TEST 3: PO structure (no comparison) â€” should be VALID (prognosis)
q = "In elderly patients with heart failure, what is the 5-year mortality rate?"
print(f"Q: {q}\n")
print(ask(PICO_Rules, q))

Q: In elderly patients with heart failure, what is the 5-year mortality rate?

P: elderly patients with heart failure
I: <MISSING>
C: <MISSING>
O: 5-year mortality rate

Status: PO

Missing: I, C


In [21]:
# TEST 4: Non-clinical question â€” should be INVALID
q = "What is the capital of France?"
print(f"Q: {q}\n")
print(ask(PICO_Rules, q))

Q: What is the capital of France?

P:
I:
C:
O:

Status: INVALID

Missing: P, I, O


In [ ]:
# TEST 5: Vague â€” should NOT infer missing elements
q = "In diabetic patients, what happens?"
print(f"Q: {q}\n")
print(ask(PICO_Rules, q))

---
## Prompt 3 â€” PMC_identifier (Link Extractor)

In [42]:
PMC_identifier = """
You are a PubMed Central (PMC) article identifier.

Given a piece of text from a medical article, your job is to identify which PMC article it came from.

Steps:
1. Look for any explicit PMC ID in the text (formats: "PMC1065064", "PMCID: 1065064", chunk IDs like "1-PMC1065064-p3-c1")
2. If no explicit ID exists, use the article title, authors, abstract content, or journal name to identify the article from your knowledge
3. If you can identify the article, return:
   PMC_ID: PMCXXXXXXX
   URL: https://pmc.ncbi.nlm.nih.gov/articles/PMCXXXXXXX/
   Confidence: High / Medium / Low
4. If you cannot identify it at all, return exactly: NULL

Rules:
- Never fabricate a PMC ID you are not confident about
- If confidence is Low, still return the best guess but mark it clearly
- Do not add any extra text beyond the format above
"""


def extract_pmc(paragraph: str) -> str:
    return ask(PMC_identifier, paragraph)

In [39]:
text = "The study details are here https://pmc.ncbi.nlm.nih.gov/articles/PMC1234567/ for reference."
print("Output:", extract_pmc(text))

Output: PMC_ID: PMC1234567
URL: https://pmc.ncbi.nlm.nih.gov/articles/PMC1234567/


In [34]:
text = "See https://pmc.ncbi.nlm.nih.gov/articles/PMC1111111/ and https://pmc.ncbi.nlm.nih.gov/articles/PMC9999999/"
print("Output:", extract_pmc(text))

Output: https://pmc.ncbi.nlm.nih.gov/articles/PMC1111111/,https://pmc.ncbi.nlm.nih.gov/articles/PMC9999999/


In [43]:
text = """TITLE:
Prospective evaluation of an internet-linked handheld computer critical care knowledge access 
system
ABSTRACT:
INTRODUCTION: Critical care physicians may benefit from immediate access to medical 
reference material. We evaluated the feasibility and potential benefits of a handheld computer 
based knowledge access system linking a central academic intensive care unit (ICU) to multiple 
community-based ICUs. METHODS: Four community hospital ICUs with 17 physicians 
participated in this prospective interventional study. Following training in the use of an internet
linked, updateable handheld computer knowledge access system, the physicians used the 
handheld devices in their clinical environment for a 12-month intervention period. Feasibility of the 
system was evaluated by tracking use of the handheld computer and by conducting surveys and 
focus group discussions. Before and after the intervention period, participants underwent 
simulated patient care scenarios designed to evaluate the information sources they accessed, as 
well as the speed and quality of their decision making. Participants generated admission orders 
during each scenario, which were scored by blinded evaluators. RESULTS: Ten physicians (59%) 
used the system regularly, predominantly for nonmedical applications (median 32.8/month, 
interquartile range [IQR] 28.3–126.8), with medical software accessed less often (median 9/month, 
IQR 3.7–13.7). Eight out of 13 physicians (62%) who completed the final scenarios chose to use 
the handheld computer for information access. The median time to access information on the 
handheld handheld computer was 19 s (IQR 15–40 s). This group exhibited a significant 
improvement in admission order score as compared with those who used other resources (P = 
0.018). Benefits and barriers to use of this technology were identified. CONCLUSION: An 
updateable handheld computer system is feasible as a means of point-of-care access to medical 
reference material and may improve clinical decision making. However, during the study, 
acceptance of the system was variable. Improved training and new technology may overcome 
some of the barriers we identified"""
print("Output:", extract_pmc(text))

Output: PMC_ID: PMC1065064
URL: https://pmc.ncbi.nlm.nih.gov/articles/PMC1065064/
Confidence: High


In [36]:
text = "Visit ncbi.nlm.nih.gov for more info. Also see pubmed.ncbi.nlm.nih.gov/12345"
print("Output:", extract_pmc(text))

Output: NULL


---
## Neo4j Live Test â€” Real GraphRAG Context
Connects to your running Neo4j instance and runs a real query through the pipeline.
Make sure Neo4j is running before executing these cells.

In [12]:
import sys
import tqdm
sys.path.insert(0, "..")

from api.services.graph_rag_service import GraphRAGService

print("GraphRAGService imported successfully.")

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GraphRAGService imported successfully.


In [13]:
# Connect to Neo4j and run a real query
# Change the query below to any medical topic you have ingested

QUERY = "What is hypertension?"

with GraphRAGService() as service:
    answer = service.perform_graph_rag(QUERY)

print("Answer:")
print(answer)


[Step 1] Embedding query and performing vector search...
[Step 2] Traversing graph for related entities and facts...


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    CALL db.index.vector.queryNodes('chunk_embeddings', $top_k, $query_embedding)\n    YIELD node AS chunk, score\n    MATCH (chunk)-[:MENTIONS]->(e:Entity)\n    OPTIONAL MATCH (e)-[r:RELATED_TO]-(neighbor:Entity)\n    RETURN \n        collect(DISTINCT chunk.text) AS chunk_contexts,\n        collect(DISTINCT {\n            name: e.name, \n

[Step 3] Fusing vector and graph context...
[Step 4] Generating final answer from LLM...
Answer:
The provided context does not include a direct definition or detailed information about **hypertension**. However, based on general medical knowledge:

**Hypertension** (commonly known as high blood pressure) is a chronic medical condition characterized by persistently elevated pressure in the arteries. It means the force of the blood against the artery walls is consistently too high, which can lead to health problems such as heart disease, stroke, and kidney damage.

Key points about hypertension:

- It is often called the "silent killer" because it may have no obvious symptoms but can cause significant organ damage over time.

- Blood pressure is measured using two numbers: systolic pressure (pressure when the heart beats) and diastolic pressure (pressure when the heart rests between beats). Hypertension is generally diagnosed when these values are consistently above normal thresholds (e.

In [14]:
# Inspect what Neo4j actually retrieves (raw context before LLM)
# Useful for checking if the right chunks/entities are being pulled

from api.repositories.neo4j_repsitory import Neo4jRepository
from api.services.text_chunker import TextChunker

chunker = TextChunker()
neo4j   = Neo4jRepository()

QUERY = "What is hypertension?"
embedding = chunker.compute_embeddings([QUERY])[0]
context   = neo4j.retrieve_hybrid_context(embedding, top_k=5)

print("=== CHUNKS ===")
for c in context["chunk_contexts"]:
    print("-", c[:200])

print("\n=== ENTITIES ===")
for e in context["entities"]:
    print(f"  {e['name']} ({e['type']}): {e['description'][:100]}")

print("\n=== RELATIONSHIPS ===")
for r in context["relationships"]:
    print(f"  {r['source']} â†’ {r['target']}")

neo4j.close()

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    CALL db.index.vector.queryNodes('chunk_embeddings', $top_k, $query_embedding)\n    YIELD node AS chunk, score\n    MATCH (chunk)-[:MENTIONS]->(e:Entity)\n    OPTIONAL MATCH (e)-[r:RELATED_TO]-(neighbor:Entity)\n    RETURN \n        collect(DISTINCT chunk.text) AS chunk_contexts,\n        collect(DISTINCT {\n            name: e.name, \n

=== CHUNKS ===
- s a common condition in critically ill patients, and RBC transfusions are often used in the treatment and management of this patient population. In fact, one study reported that 25% of all critically 
- edict mortality before, during, and after critical care transport. Limited physiologic parameters available on transport (i.e. pulse, blood pressure, respiratory rate, and GCS) were used and scored nu
- he attending physicians who recruited patients into the study did not enter those patients who were considered to have severe cardiac disease. HÃ©bert et al. sought to examine further whether a restric
- rdiac disease and similar changes in haemoglobin levels showed no increase in mortality, which is in accordance with the results of the TRICC trial. In the study by HÃ©bert et al. of 4470 critically il
- abnormality on HRCT). Honeycombing and bronchiectatic changes were found in only 3 SARS patients with high HRCT score (5.5 Â± 2.7% of total abnormality on HRCT). Seven o

In [15]:
# Test Rule prompt WITH real Neo4j context injected
# This simulates exactly what will happen when Rule is wired into graph_rag_service.py

chunker = TextChunker()
neo4j   = Neo4jRepository()

QUERY     = "What is hypertension?"
embedding = chunker.compute_embeddings([QUERY])[0]
context   = neo4j.retrieve_hybrid_context(embedding, top_k=5)
neo4j.close()

chunks    = "\n".join([f"- {c}" for c in context["chunk_contexts"]])
entities  = "\n".join([f"- {e['name']} ({e['type']}): {e['description']}" for e in context["entities"]])

user_message = f"""
[RETRIEVED CONTEXT]
{chunks}
{entities}
[END CONTEXT]

{QUERY}
"""

print("Answer with real context:")
print(ask(Rule, user_message))

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    CALL db.index.vector.queryNodes('chunk_embeddings', $top_k, $query_embedding)\n    YIELD node AS chunk, score\n    MATCH (chunk)-[:MENTIONS]->(e:Entity)\n    OPTIONAL MATCH (e)-[r:RELATED_TO]-(neighbor:Entity)\n    RETURN \n        collect(DISTINCT chunk.text) AS chunk_contexts,\n        collect(DISTINCT {\n            name: e.name, \n

Answer with real context:
Not found in the knowledge base.


---
## Option 3 — Context-Driven (No Classifier)
Retrieved context shapes the answer. PICO-structured PMC chunks → PICO answer. General chunks → general answer. No routing needed.

In [ ]:
# Option 3 — Smart prompt that uses context as the signal
# The prompt detects PICO structure from the retrieved chunks itself

Rule_v2 = """
--Role--
You are an Evidence-Based Medical Chatbot for doctors and researchers.
You answer ONLY from the retrieved context provided. Never use your own knowledge.

--Context Requirement--
Every medical query will include:

[RETRIEVED CONTEXT]
<chunks, entities, relationships from Neo4j>
[END CONTEXT]

If this block is ABSENT or EMPTY respond exactly: "Not found in the knowledge base."

--Answer Format (auto-detect from context)--
Look at the retrieved context:

IF the context contains structured clinical trial data with Population, Intervention, Comparison or Outcome information:
  - Answer in PICO format:
    P (Population):
    I (Intervention):
    C (Comparison):
    O (Outcome):
    Evidence Summary:

IF the context is general medical/explanatory text:
  - Answer in clear paragraphs, bold key terms, be concise.

--Rules--
1. Never infer beyond what the context states.
2. Never fabricate PMC IDs. If IDs appear in context, cite them. Otherwise: "Sources: https://pmc.ncbi.nlm.nih.gov/"
3. Never provide diagnosis or treatment recommendations.
4. End every answer with: Coverage: Complete / Partial / Insufficient data
5. For greetings or off-topic chat, respond briefly. No Coverage label.
"""

print("Rule_v2 defined.")

In [ ]:
# TEST — General question (no PICO structure expected in context)
from api.repositories.neo4j_repsitory import Neo4jRepository
from api.services.text_chunker import TextChunker

chunker = TextChunker()
neo4j   = Neo4jRepository()

QUERY     = "What is hypertension?"
embedding = chunker.compute_embeddings([QUERY])[0]
context   = neo4j.retrieve_hybrid_context(embedding, top_k=5)
neo4j.close()

chunks   = "
".join([f"- {c}" for c in context["chunk_contexts"]])
entities = "
".join([f"- {e['name']} ({e['type']}): {e['description']}" for e in context["entities"]])

user_msg = f"""
[RETRIEVED CONTEXT]
{chunks}
{entities}
[END CONTEXT]

{QUERY}
"""

print(f"Q: {QUERY}")
print(ask(Rule_v2, user_msg))

In [ ]:
# TEST — Clinical PICO question (expects PICO format if context has trial data)
chunker = TextChunker()
neo4j   = Neo4jRepository()

QUERY     = "In adults with hypertension, does ACE inhibitor treatment compared to placebo reduce cardiovascular events?"
embedding = chunker.compute_embeddings([QUERY])[0]
context   = neo4j.retrieve_hybrid_context(embedding, top_k=5)
neo4j.close()

chunks   = "
".join([f"- {c}" for c in context["chunk_contexts"]])
entities = "
".join([f"- {e['name']} ({e['type']}): {e['description']}" for e in context["entities"]])

user_msg = f"""
[RETRIEVED CONTEXT]
{chunks}
{entities}
[END CONTEXT]

{QUERY}
"""

print(f"Q: {QUERY}")
print(ask(Rule_v2, user_msg))

---
## Classifier Test — Query Routing
A lightweight LLM call classifies the query before GraphRAG runs.
Returns {"type": "pico"} or {"type": "general"} then routes to the correct prompt.

In [22]:
import json as _json

CLASSIFIER_PROMPT = """
You are a medical query classifier.

Classify the user query into exactly one of two types:
- "pico"    : A structured clinical question with identifiable Population, Intervention, and Outcome
- "general" : A general medical question, definition, explanation, or non-clinical query

Respond ONLY with valid JSON, nothing else:
{"type": "pico"} or {"type": "general"}
"""

def classify_query(query: str) -> str:
    raw = ask(CLASSIFIER_PROMPT, query)
    try:
        result = _json.loads(raw)
        return result.get("type", "general")
    except Exception:
        return "general"

print("Classifier defined.")

Classifier defined.


In [23]:
# Run classifier on various queries and see how it routes them

test_queries = [
    "What is hypertension?",
    "In adults with type 2 diabetes, does metformin vs insulin improve HbA1c?",
    "Who discovered penicillin?",
    "In elderly patients with heart failure, does beta-blocker therapy reduce mortality compared to placebo?",
    "What are the side effects of aspirin?",
    "In children with asthma aged 5-12, does inhaled corticosteroid improve FEV1 outcomes?",
    "Hello, how are you?",
]

print(f"{'Query':<65} | Type")
print("-" * 75)
for q in test_queries:
    qtype = classify_query(q)
    print(f"{q[:64]:<65} | {qtype}")

Query                                                             | Type
---------------------------------------------------------------------------
What is hypertension?                                             | general
In adults with type 2 diabetes, does metformin vs insulin improv  | pico
Who discovered penicillin?                                        | general
In elderly patients with heart failure, does beta-blocker therap  | pico
What are the side effects of aspirin?                             | general
In children with asthma aged 5-12, does inhaled corticosteroid i  | pico
Hello, how are you?                                               | general


---
## 3-Class Guardrail Test
Classifies queries into: `general` / `casual_medical` / `clinical`
Only `clinical` passes through to GraphRAG.

In [24]:
import json as _json

GUARDRAIL_PROMPT = """
You are a medical query guardrail classifier.

Classify the user query into exactly one of three categories:

1. "general"
   - Completely off-topic, no real medical relevance
   - Casual conversation or random mentions of medical words without clinical intent
   - Examples: "who won the cricket world cup?", "I ate cheese and got diarrhea maybe cancer lol"

2. "casual_medical"
   - The user is describing a personal symptom or asking for general health advice
   - The question is about themselves, not about clinical evidence or research
   - Key signal: first-person complaints ("I have", "I feel", "I ate")
   
3. "clinical"
   - The user is asking for clinical evidence, research outcomes, or medical literature
   - The question is about patient populations, interventions, or outcomes
   - Key signal: third-person or population-level framing


Respond ONLY with valid JSON, nothing else:
{"class": "general"} or {"class": "casual_medical"} or {"class": "clinical"}
"""

def guardrail(query: str) -> str:
    raw = ask(GUARDRAIL_PROMPT, query)
    try:
        result = _json.loads(raw.strip())
        return result.get("class", "general")
    except Exception:
        return "general"

print("3-class guardrail defined.")


3-class guardrail defined.


In [25]:
test_queries = [
    # Should be GENERAL
    ("Who won the cricket world cup?",                                           "general"),
    ("I ate cheese now I am getting diarrhea and I will get cancer maybe",       "general"),
    ("What is 2 + 2?",                                                           "general"),
    ("Tell me a joke",                                                            "general"),

    # Should be CASUAL_MEDICAL
    ("I have a headache what should I do?",                                      "casual_medical"),
    ("Is paracetamol safe to take daily?",                                       "casual_medical"),
    ("What foods should I avoid if I have high blood pressure?",                 "casual_medical"),
    ("I feel tired all the time, am I sick?",                                    "casual_medical"),

    # Should be CLINICAL
    ("Does metformin reduce HbA1c levels in adults with type 2 diabetes?",       "clinical"),
    ("What is the 5-year mortality rate of heart failure in elderly patients?",   "clinical"),
    ("In hypertensive patients, does ACE inhibitor reduce cardiovascular risk?",  "clinical"),
    ("What does PMC say about beta-blocker therapy post myocardial infarction?",  "clinical"),
    ("What is hypertension according to PMC articles?",                           "clinical"),
]

correct = 0
print(f"{'Query':<62} | {'Expected':<15} | {'Got':<15} | Result")
print("-" * 103)
for query, expected in test_queries:
    got = guardrail(query)
    match = got == expected
    if match:
        correct += 1
    status = "PASS" if match else "FAIL"
    print(f"{query[:61]:<62} | {expected:<15} | {got:<15} | {status}")

total = len(test_queries)
print(f"\nAccuracy: {correct}/{total} ({100*correct//total}%)")


Query                                                          | Expected        | Got             | Result
-------------------------------------------------------------------------------------------------------
Who won the cricket world cup?                                 | general         | general         | PASS
I ate cheese now I am getting diarrhea and I will get cancer   | general         | general         | PASS
What is 2 + 2?                                                 | general         | general         | PASS
Tell me a joke                                                 | general         | general         | PASS
I have a headache what should I do?                            | casual_medical  | casual_medical  | PASS
Is paracetamol safe to take daily?                             | casual_medical  | casual_medical  | PASS
What foods should I avoid if I have high blood pressure?       | casual_medical  | casual_medical  | PASS
I feel tired all the time, am I sick?         

---
## 4-Class Guardrail v2
Adds `case_description` class for patient scenarios without a researchable question.
The chatbot asks for clarification instead of running GraphRAG or rejecting.

In [26]:
import json as _json

GUARDRAIL_V2_PROMPT = """
You are a medical query guardrail classifier.

Classify the user query into exactly one of four categories:

1. "general"
   - Completely off-topic with no meaningful medical intent
   - Casual conversation, jokes, or accidental mentions of medical words without any clinical context
   - Examples:
     * "Who won the cricket world cup?"
     * "I ate cheese now I am getting diarrhea and maybe I will get cancer lol"
     * "Tell me a joke"

2. "casual_medical"
   - A personal, first-person minor health complaint or lifestyle health question
   - The user is describing themselves and wants simple advice
   - Does NOT have a researchable question — no population, no intervention, no outcome being asked
   - Key signal: first-person ("I have", "I feel", "I am", "should I")
   - Examples:
     * "I have a headache, what should I do?"
     * "Is it okay to take paracetamol every day?"
     * "I feel tired all the time"
     * "What foods should I avoid for high blood pressure?"

3. "case_description"
   - Describes a patient scenario (could be first or third person) without asking a specific researchable question
   - Contains patient details (age, gender, symptoms, drugs) but no clear clinical question
   - The user is presenting a case, not asking for evidence
   - Key signal: describes WHO the patient is and WHAT is happening, but does not ask WHAT THE EVIDENCE SAYS
   - Examples:
     * "A 28 year old male has consistent headache, takes paracetamol but cannot sleep at night"
     * "My patient is a 60 year old diabetic woman on insulin with high BP"
     * "A child aged 5 has been wheezing for 3 days and has a fever"

4. "clinical"
   - A proper evidence-based medical or clinical research question
   - Asks for outcomes, evidence, effectiveness, mortality, or clinical findings
   - Has a researchable question even if not fully in PICO format
   - Key signal: asks WHAT THE EVIDENCE or RESEARCH SAYS about a population or intervention
   - Examples:
     * "Does metformin reduce HbA1c in adults with type 2 diabetes?"
     * "What is the 5-year mortality rate of heart failure in elderly patients?"
     * "In hypertensive patients, does ACE inhibitor reduce cardiovascular events compared to placebo?"
     * "What does PMC say about beta-blocker therapy after myocardial infarction?"

Respond ONLY with valid JSON, nothing else:
{"class": "general"} or {"class": "casual_medical"} or {"class": "case_description"} or {"class": "clinical"}
"""

def guardrail_v2(query: str) -> str:
    raw = ask(GUARDRAIL_V2_PROMPT, query)
    try:
        result = _json.loads(raw.strip())
        return result.get("class", "general")
    except Exception:
        return "general"

# Routing explanation
ROUTING = {
    "general":          "Reject — off topic",
    "casual_medical":   "Short friendly reply — no GraphRAG",
    "case_description": "Ask user: what specifically do you want to know about this case?",
    "clinical":         "Full GraphRAG pipeline",
}

print("4-class guardrail v2 defined.")


4-class guardrail v2 defined.


In [27]:
test_queries = [
    # GENERAL
    ("Who won the cricket world cup?",                                                        "general"),
    ("I ate cheese now I am getting diarrhea maybe I will get cancer lol",                   "general"),
    ("Tell me a joke",                                                                         "general"),
    ("What is 2 + 2?",                                                                         "general"),

    # CASUAL_MEDICAL
    ("I have a headache what should I do?",                                                   "casual_medical"),
    ("Is paracetamol safe to take every day?",                                                "casual_medical"),
    ("I feel tired all the time, am I sick?",                                                 "casual_medical"),
    ("What foods should I avoid if I have high blood pressure?",                              "casual_medical"),

    # CASE_DESCRIPTION
    ("A 28 year old male has consistent headache, takes paracetamol but cannot sleep",       "case_description"),
    ("My patient is a 60 year old diabetic woman on insulin with high blood pressure",       "case_description"),
    ("A child aged 5 has been wheezing for 3 days and has a fever",                          "case_description"),
    ("45 year old male smoker presents with chest pain and shortness of breath",             "case_description"),

    # CLINICAL
    ("Does metformin reduce HbA1c levels in adults with type 2 diabetes?",                   "clinical"),
    ("What is the 5-year mortality rate of heart failure in elderly patients?",               "clinical"),
    ("In hypertensive patients does ACE inhibitor reduce cardiovascular risk vs placebo?",    "clinical"),
    ("What does PMC say about beta-blocker therapy post myocardial infarction?",              "clinical"),
    ("What is hypertension according to PMC articles?",                                       "clinical"),
]

correct = 0
print(f"{'Query':<65} | {'Expected':<16} | {'Got':<16} | Result")
print("-" * 110)
for query, expected in test_queries:
    got = guardrail_v2(query)
    match = got == expected
    if match:
        correct += 1
    status = "PASS" if match else "FAIL"
    route = ROUTING.get(got, "unknown")
    print(f"{query[:64]:<65} | {expected:<16} | {got:<16} | {status}")

total = len(test_queries)
print(f"\nAccuracy: {correct}/{total} ({100*correct//total}%)")
print("\n--- Routing reference ---")
for cls, action in ROUTING.items():
    print(f"  {cls:<18} -> {action}")


Query                                                             | Expected         | Got              | Result
--------------------------------------------------------------------------------------------------------------
Who won the cricket world cup?                                    | general          | general          | PASS
I ate cheese now I am getting diarrhea maybe I will get cancer l  | general          | general          | PASS
Tell me a joke                                                    | general          | general          | PASS
What is 2 + 2?                                                    | general          | general          | PASS
I have a headache what should I do?                               | casual_medical   | casual_medical   | PASS
Is paracetamol safe to take every day?                            | casual_medical   | casual_medical   | PASS
I feel tired all the time, am I sick?                             | casual_medical   | casual_medical   | PASS